In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

In [23]:
df = pd.read_csv('Housing Price/housing.csv')

In [24]:
df = df.sample(n=2000, random_state=42)

In [25]:
df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].median()) # chosen median because it is not affected by outliers

In [26]:
# 1. Define Column Groups
numeric_cols = [
    'longitude', 'latitude', 'housing_median_age',
    'total_rooms', 'total_bedrooms', 'population',
    'households', 'median_income'
]
categorical_cols = ['ocean_proximity']

# 2. Build Pipeline for Numeric Features
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 3. Build Pipeline for Categorical Features
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# 4. Combine Transformers into ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numeric_cols),
        ('cat', cat_pipeline, categorical_cols)
    ]
)

# 5. Build Final End-to-End Pipeline
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

#### Preprocessing & Modeling Pipeline

End-to-end `sklearn` pipeline handling missing values, scaling, and
encoding — fit only on training data during CV to avoid leakage.

**Numeric** (`longitude`, `latitude`, `housing_median_age`,
`total_rooms`, `total_bedrooms`, `population`, `households`,
`median_income`): median imputation → `StandardScaler`.

**Categorical** (`ocean_proximity`): most-frequent imputation →
`OneHotEncoder(handle_unknown='ignore')`.

**Combine:** `ColumnTransformer` routes each column group to its
pipeline.

**Full pipeline:** `preprocessor` → `RandomForestRegressor(random_state=42)`,
so `full_pipeline.fit(X_train, y_train)` handles preprocessing and
modeling in one call, refitting transforms per CV fold.

In [27]:
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

In [28]:
# Usage Example: Fit and Predict without Data Leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

full_pipeline.fit(X_train, y_train)
predictions = full_pipeline.predict(X_test)

In [29]:
# 1. Number of rooms per household
df['rooms_per_household'] = df['total_rooms'] / df['households']

# 2.Number of bedrooms per room (ratio of rooms allocated for sleeping)
df['bedrooms_per_room'] = df['total_bedrooms'] / df['total_rooms']

# 3. Number of people per household (average household size)
df['population_per_household'] = df['population'] / df['households']


In [30]:
# Split Data
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define Hyperparameter Search Grid
param_grid = {
    'preprocessor__num__imputer__strategy': ['median', 'mean'],
    'model__n_estimators': [100, 200],
    'model__max_depth': [15, 25, None],
    'model__min_samples_split': [2, 5]
}

# Run 5-Fold GridSearchCV (Fixed: uses X_train)
grid_search = GridSearchCV(
    estimator=full_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

# Evaluate Best Model
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("Best Parameters:", grid_search.best_params_)
print(f"Test RMSE: ${root_mean_squared_error(y_test, y_pred):,.2f}")
print(f"Test R²  : {r2_score(y_test, y_pred):.4f}")

Best Parameters: {'model__max_depth': None, 'model__min_samples_split': 5, 'model__n_estimators': 100, 'preprocessor__num__imputer__strategy': 'median'}
Test RMSE: $60,078.13
Test R²  : 0.7390


#### Full Pipeline Hyperparameter Search & Test Evaluation

Split the raw (unscaled, unimputed) data 80/20 — all preprocessing is
now handled inside `full_pipeline`, so `X_train`/`X_test` stay raw here.

**Grid:** 2 imputer strategies × 2 `n_estimators` × 3 `max_depth` ×
2 `min_samples_split` = 24 combinations, searched with `GridSearchCV`
(5-fold, `neg_root_mean_squared_error`, `n_jobs=-1`).

Unlike the earlier standalone RF search, this tunes both **preprocessing**
(imputer strategy) and **model** hyperparameters jointly, and is fit on
`X_train` — the previous version's bug (fitting on the full `X`) is
fixed here.

**Final evaluation:** best pipeline refit on all of `X_train`, then
scored once on the held-out `X_test` — reporting **Test RMSE** and
**Test R²** as the final, unbiased performance estimate.

**Best Parameters:** `{'model__max_depth': None, 'model__min_samples_split': 5, 'model__n_estimators': 100, 'preprocessor__num__imputer__strategy': 'median'}`

| Metric | Value |
|---|---|
| Test RMSE | $60,078.13 |
| Test R² | 0.7390 |

The search settled on unrestricted tree depth (`max_depth=None`) with a
higher `min_samples_split=5` — suggesting the trees compensate for going
fully deep by requiring more samples before splitting, a mild built-in
regularization rather than an explicit depth cap. `median` imputation
edged out `mean`, likely because income and room/bedroom counts are
right-skewed, where the median is less distorted by outliers. An R² of
0.739 means the model explains ~74% of the variance in house prices on
data it never saw during training.

In [31]:
# Train Untuned Baseline Model for Comparison
baseline_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=50, random_state=42))
])

baseline_pipeline.fit(X_train, y_train)
baseline_preds = baseline_pipeline.predict(X_test)

# Calculate Baseline Metrics
baseline_rmse = root_mean_squared_error(y_test, baseline_preds)
baseline_mae = mean_absolute_error(y_test, baseline_preds)
baseline_r2 = r2_score(y_test, baseline_preds)

# Evaluate Tuned Model (Best Estimator from GridSearchCV)
best_pipeline = grid_search.best_estimator_
tuned_preds = best_pipeline.predict(X_test)

# Calculate Tuned Metrics
tuned_rmse = root_mean_squared_error(y_test, tuned_preds)
tuned_mae = mean_absolute_error(y_test, tuned_preds)
tuned_r2 = r2_score(y_test, tuned_preds)

# Print Final Comparison Report
rmse_improvement = ((baseline_rmse - tuned_rmse) / baseline_rmse) * 100
print("-" * 60)
print("HELD-OUT TEST SET EVALUATION REPORT")
print("-" * 60)
print(f"{'Metric':<15} | {'Baseline Model':<18} | {'Tuned Pipeline':<18}")
print("-" * 60)
print(f"{'RMSE ($)':<15} | {baseline_rmse:<18,.2f} | {tuned_rmse:<18,.2f}")
print(f"{'MAE ($)':<15} | {baseline_mae:<18,.2f} | {tuned_mae:<18,.2f}")
print(f"{'R² Score':<15} | {baseline_r2:<18.4f} | {tuned_r2:<18.4f}")
print("-" * 60)
print(f"Total RMSE Improvement: {rmse_improvement:.2f}% reduction in prediction error")
print("-" * 60)

------------------------------------------------------------
HELD-OUT TEST SET EVALUATION REPORT
------------------------------------------------------------
Metric          | Baseline Model     | Tuned Pipeline    
------------------------------------------------------------
RMSE ($)        | 60,433.32          | 60,078.13         
MAE ($)         | 41,861.78          | 41,928.54         
R² Score        | 0.7359             | 0.7390            
------------------------------------------------------------
Total RMSE Improvement: 0.59% reduction in prediction error
------------------------------------------------------------


#### Baseline vs. Tuned Pipeline — Held-Out Test Comparison

Trained an **untuned baseline** pipeline (`n_estimators=50`, default
depth) alongside the **tuned pipeline** (`grid_search.best_estimator_`),
both evaluated on the same held-out `X_test` for a direct, apples-to-apples
comparison.

**Metrics computed for each:** RMSE, MAE, and R²
(`root_mean_squared_error`, `mean_absolute_error`, `r2_score`).

**Results:**

| Metric | Baseline Model | Tuned Pipeline |
|---|---|---|
| RMSE ($) | 60,433.32 | 60,078.13 |
| MAE ($) | 41,861.78 | 41,928.54 |
| R² Score | 0.7359 | 0.7390 |

**Total RMSE Improvement:** 0.59% reduction in prediction error.

Consistent with the CV-based comparison earlier, tuning produced only a
marginal RMSE gain on the truly held-out test set — 0.59% is a small,
practically negligible improvement. Notably, **MAE slightly worsened**
(41,861.78 → 41,928.54) with tuning, while RMSE improved — since RMSE
penalizes large errors more heavily than MAE, this suggests the tuned
model got a bit better at reducing a few large errors (e.g. on
high-value outlier homes) at the cost of being marginally less accurate
on typical predictions. Overall, this confirms the earlier finding: the
untuned baseline was already close to this ceiling, and hyperparameter
tuning within this search space offered limited additional value.